In [6]:
!pip freeze | grep 'streamlit-jupyter @ git' >/dev/null || { pip uninstall -y --quiet streamlit-jupyter 2>/dev/null; pip install --quiet git+https://github.com/ddobrinskiy/streamlit-jupyter.git 2>/dev/null; }

In [9]:
import logging
import pandas as pd
import altair as alt
import streamlit as st
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# basics page
st.set_page_config(layout="wide")
st.title("Analyse Ärztebewertungen")

# Datenladen
def load_data():
    doc = pd.read_csv(
        "project_files/doc_extended.csv",
        sep=";",
        encoding="utf-8"
    )
    rev = pd.read_csv(
        "project_files/rev_cleaned.csv",
        encoding="utf-8"
    )

    df = doc.merge(
        rev,
        left_on="arzt_id",
        right_on="ref_id",
        how="left"
    )
    df["region"] = df["bundesland"]
    return df

typ_map = pd.read_csv(
    "project_files/typ.csv",
    sep=";",
    encoding="latin1"
)

typ_map = typ_map.rename(columns={
    "typ_id": "typ_id",
    "typ": "typ_name"
})

typ_map["typ_col"] = "typ_" + typ_map["typ_id"].astype(str)

df_base = load_data()
st.success("Daten erfolgreich geladen")

#navi
page = st.sidebar.radio(
    "Auswertung auswählen",
    [
        "Überblick",
        "Fachrichtungen",
        "Regionen / Städte",
        "Versicherungsstatus",
        "Stadtgröße & Ärzte",
        "Bewertungskriterien",
        "Verteilungen & Streuung",
        "Arzt-Profiling",
        "Uneinigkeit der Kriterien",
        "Median der Kriterien"
    ]
)

# filter sidebar
st.sidebar.header("Filter")

region_filter = st.sidebar.multiselect(
    "Region auswählen",
    sorted(df_base["region"].dropna().unique())
)


# Region - Stadt dependency
if region_filter:
    stadt_options = (
        df_base[df_base["region"].isin(region_filter)]["stadt"]
        .dropna()
        .unique()
    )
else:
    stadt_options = df_base["stadt"].dropna().unique()

stadt_filter = st.sidebar.multiselect(
    "Stadt auswählen",
    sorted(stadt_options)
)

# fachgruppe
fachg_filter = st.sidebar.multiselect(
    "Fachgruppe auswählen",
    sorted(df_base["fachgruppe"].dropna().unique())
)


# Fachgruppe - Fachrichtung Dependance 
if fachg_filter:
    fachr_options = (
        df_base[df_base["fachgruppe"].isin(fachg_filter)]["fachrichtung"]
        .dropna()
        .unique()
    )
else:
    fachr_options = df_base["fachrichtung"].dropna().unique()

fachr_filter = st.sidebar.multiselect(
    "Fachrichtung auswählen",
    sorted(fachr_options)
)

# grades
note_range = st.sidebar.slider(
    "Notenbereich auswählen",
    1.0, 6.0, (1.0, 6.0), 0.1
)

min_bewertungen_arzt = st.sidebar.slider(
    "Mindestanzahl Bewertungen pro Arzt",
    min_value=1,
    max_value=100,
    value=5,
    step=1
)

#filter anwenden
df = df_base.copy()

if region_filter:
    df = df[df["region"].isin(region_filter)]

if stadt_filter:
    df = df[df["stadt"].isin(stadt_filter)]

if fachg_filter:
    df = df[df["fachgruppe"].isin(fachg_filter)]

if fachr_filter:
    df = df[df["fachrichtung"].isin(fachr_filter)]

df = df[df["gesamt_note"].between(*note_range)]

arzt_counts = (
    df.groupby("arzt_id")
      .size()
      .reset_index(name="anzahl_bewertungen")
)

gueltige_aerzte = arzt_counts.loc[
    arzt_counts["anzahl_bewertungen"] >= min_bewertungen_arzt,
    "arzt_id"
]

df = df[df["arzt_id"].isin(gueltige_aerzte)]


#catch issues
if df.empty:
    st.warning("Keine Daten für die aktuelle Filterkombination.")
    st.stop()

if page == "Überblick":
    st.subheader("Kennzahlen")

    c1, c2, c3 = st.columns(3)
    
    with c1:
        st.metric("Anzahl Bewertungen", len(df))

    with c2:
        st.metric("Ø Gesamtnote", round(df["gesamt_note"].mean(), 2))
    
    with c3:
        st.metric("Anzahl Ärzte", df["arzt_id"].nunique())

    
    st.divider()
    
    st.subheader("Beispieldaten")
    st.dataframe(df.head(20), use_container_width=True)


#Werden die Arzt-Typen im Durchschnitt alle gleich bewertet?
elif page == "Fachrichtungen":
    st.subheader("Werden Fachrichtungen gleich bewertet?")

    fachgrp_avg = (
        df.groupby("fachgruppe")
        .agg(
            avg_note=("gesamt_note", "mean"),
            anzahl_bewertungen=("gesamt_note", "count")
        )
        .reset_index()
        .sort_values("avg_note")
    )

    st.dataframe(fachgrp_avg)
    st.bar_chart(fachgrp_avg.set_index("fachgruppe")["avg_note"])

#Gibt es Regionen, in denen besonders gut/schlecht bewertet wird?
elif page == "Regionen / Städte":
    st.subheader("Regionale Unterschiede")

    city_counts = df["stadt"].value_counts()
    valid_cities = city_counts[city_counts >= 50].index

    avg_city = (
        df[df["stadt"].isin(valid_cities)]
        .groupby("stadt", as_index=False)
        .agg(avg_note=("gesamt_note", "mean"))
        .sort_values("avg_note")
    )

    st.subheader("Städte (≥ 50 Bewertungen)")
    st.dataframe(avg_city)
    st.bar_chart(avg_city.set_index("stadt")["avg_note"])

    st.subheader("Regionen")

    avg_region = (
        df.groupby("region", as_index=False)
        .agg(avg_note=("gesamt_note", "mean"))
        .sort_values("avg_note")
    )

    st.dataframe(avg_region)
    st.bar_chart(avg_region.set_index("region")["avg_note"])

# Gibt es einen Unterschied in der Bewertung von Kassen- und Privatpatienten?
elif page == "Versicherungsstatus":
    st.subheader("Bewertung nach Versicherungsstatus")

    avg_kasse = (
        df.groupby("kasse_privat", as_index=False)
        .agg(avg_note=("gesamt_note", "mean"))
        .sort_values("avg_note")
    )

    st.dataframe(avg_kasse)
    st.bar_chart(avg_kasse.set_index("kasse_privat")["avg_note"])

# Ist die Anzahl der gemeldeten Ärzte proportional zur Stadtgröße?
elif page == "Stadtgröße & Ärzte":
    st.subheader("Zusammenhang zwischen Stadtgröße und Anzahl gemeldeter Ärzte")

    MIN_EINWOHNER = 20000

    df_city = (
        df
        .groupby("stadt")
        .agg(
            anzahl_aerzte=("arzt_id", "nunique"),
            einwohner=("einwohner", "first")
        )
        .dropna()
        .query("einwohner >= @MIN_EINWOHNER")
        .reset_index()
    )

    chart = (
        alt.Chart(df_city)
        .mark_circle(size=80, opacity=0.7)
        .encode(
            x=alt.X("einwohner:Q", title="Einwohner"),
            y=alt.Y("anzahl_aerzte:Q", title="Anzahl gemeldeter Ärzte"),
            tooltip=[
                alt.Tooltip("stadt:N", title="Stadt"),
                alt.Tooltip("einwohner:Q", title="Einwohner", format=","),
                alt.Tooltip("anzahl_aerzte:Q", title="Ärzte")
            ]
        )
        .interactive()
    )

    st.altair_chart(chart, use_container_width=True)

# Bewertungen pro typ
elif page == "Bewertungskriterien":
    st.subheader("Bewertungskriterien (Typ 1-17)")

    # typ.csv korrekt laden
    typ_map = pd.read_csv(
        "project_files/typ.csv",
        sep=";",
        encoding="latin1"
    )

    typ_map = typ_map.rename(columns={
        "typ_id": "typ_id",
        "typ": "typ_name"
    })

    typ_map["typ_col"] = "typ_" + typ_map["typ_id"].astype(str)

    df_long = (
        df.melt(
            id_vars=["region"],
            value_vars=[c for c in df.columns if c.startswith("typ_")],
            var_name="typ_col",
            value_name="note"
        )
        .dropna()
        .merge(
            typ_map[["typ_col", "typ_name"]],
            on="typ_col",
            how="left"
        )
    )

    MIN_BEW = 100

    crit_global = (
        df_long
        .groupby("typ_name")["note"]
        .agg(avg_note="mean", anzahl="count")
        .reset_index()
        .query("anzahl >= @MIN_BEW")
        .sort_values("avg_note")
    )

    st.markdown("**Überregional: Durchschnitt je Bewertungskriterium**")
    st.dataframe(crit_global)

    st.bar_chart(
        crit_global.set_index("typ_name")["avg_note"]
    )

    st.markdown("**Top 5 (beste Bewertungen)**")
    st.dataframe(crit_global.head(5))

    st.markdown("**Bottom 5 (schlechteste Bewertungen)**")
    st.dataframe(crit_global.tail(5))

    st.subheader("Regionale Abhängigkeit der Bewertungskriterien")

    MIN_BEW = 50  # Schwelle gegen Rauschen
    
    heat = (
        df_long
        .groupby(["region", "typ_name"])
        .agg(
            avg_note=("note", "mean"),
            anzahl=("note", "count")
        )
        .reset_index()
        .query("anzahl >= @MIN_BEW")
    )
        
    heatmap = (
    alt.Chart(heat)
    .mark_rect(stroke="black", strokeWidth=0.2)
    .encode(
        x=alt.X(
            "typ_name:N",
            title="Bewertungskriterium",
            axis=alt.Axis(labelAngle=-30)
        ),
        y=alt.Y(
            "region:N",
            title="Region"
        ),
        color=alt.Color(
            "avg_note:Q",
            title="Ø Note",
            scale=alt.Scale(scheme="redyellowgreen", reverse=True)
        ),
        tooltip=[
            alt.Tooltip("typ_name:N", title="Kriterium"),
            alt.Tooltip("region:N", title="Region"),
            alt.Tooltip("avg_note:Q", title="Ø Note", format=".2f"),
            alt.Tooltip("anzahl:Q", title="Bewertungen")
        ]
    )
    .properties(height=420)
    )

    st.altair_chart(heatmap, use_container_width=True)

elif page == "Verteilungen & Streuung":
    st.subheader("Verteilung der Bewertungen")
#boxplot
    box = (
        alt.Chart(df)
        .mark_boxplot(extent="min-max")
        .encode(
            x=alt.X("fachgruppe:N", title="Fachgruppe"),
            y=alt.Y("gesamt_note:Q", title="Gesamtnote")
        )
        .properties(height=420)
    )

    st.altair_chart(box, use_container_width=True)

#Median & Standardabweichung
    stats = (
        df.groupby("fachgruppe")
        .agg(
            mean_note=("gesamt_note", "mean"),
            median_note=("gesamt_note", "median"),
            std_note=("gesamt_note", "std"),
            anzahl=("gesamt_note", "count")
        )
        .reset_index()
        .sort_values("mean_note")
    )
    
    st.subheader("Lage- und Streuungsmaße")
    st.dataframe(stats, use_container_width=True)


elif page == "Arzt-Profiling":
    st.subheader("Das große Arzt-Profiling")

    #Aggregation 
    arzt_features = (
        df.groupby("arzt_id")
        .agg(
            avg_note=("gesamt_note", "mean"),
            median_note=("gesamt_note", "median"),
            std_note=("gesamt_note", "std"),
            anzahl_bewertungen=("gesamt_note", "count")
        )
        .dropna()
        .reset_index()
    )

    # kmeans
    X = arzt_features[["avg_note", "anzahl_bewertungen"]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    k = st.slider("Anzahl Cluster (k)", 2, 5, 4)

    kmeans = KMeans(n_clusters=k, random_state=42)
    arzt_features["cluster"] = kmeans.fit_predict(X_scaled)

    st.markdown("K-Means: Arzt-Typen")

    cluster_scatter = (
        alt.Chart(arzt_features)
        .mark_circle(size=80, opacity=0.7)
        .encode(
            x=alt.X("anzahl_bewertungen:Q", title="Anzahl Bewertungen"),
            y=alt.Y("avg_note:Q", title="Ø Gesamtnote", scale=alt.Scale(reverse=True)),
            color=alt.Color("cluster:N", title="Cluster"),
            tooltip=[
                alt.Tooltip("arzt_id:N", title="Arzt-ID"),
                alt.Tooltip("avg_note:Q", title="Ø Note", format=".2f"),
                alt.Tooltip("anzahl_bewertungen:Q", title="Bewertungen"),
                alt.Tooltip("cluster:N", title="Cluster")
            ]
        )
    )

    st.altair_chart(cluster_scatter, use_container_width=True)

    # Boxplot
    st.markdown("Boxplot: Bewertungsverteilung je Cluster")

    df_cluster_long = (
        df.merge(
            arzt_features[["arzt_id", "cluster"]],
            on="arzt_id",
            how="inner"
        )
    )

    boxplot = (
        alt.Chart(df_cluster_long)
        .mark_boxplot(extent="min-max")
        .encode(
            x=alt.X("cluster:N", title="Cluster"),
            y=alt.Y("gesamt_note:Q", title="Gesamtnote")
        )
        .properties(height=420)
    )

    st.altair_chart(boxplot, use_container_width=True)

    #Median & Standardabweichung pro Cluster
    st.markdown("Median & Standardabweichung")

    cluster_stats = (
        arzt_features
        .groupby("cluster")
        .agg(
            avg_note=("avg_note", "mean"),
            median_note=("median_note", "median"),
            std_note=("std_note", "mean"),
            aerzte=("arzt_id", "count")
        )
        .reset_index()
        .sort_values("avg_note")
    )

    st.dataframe(cluster_stats, use_container_width=True)

    # Interpretation
    st.markdown("""
    **Interpretationshilfe:**
    - Niedrige Ø-Note + niedrige Std. → konsistenter Top-Arzt-Typ  
    - Hohe Std. → polarisierende Ärzte  
    - Median ≠ Mittelwert → Ausreißer-Effekte vorhanden  
    - K-Means zeigt strukturell unterschiedliche Arzt-Typen
    """)

elif page == "Uneinigkeit der Kriterien":
    st.subheader("Standardabweichung: Das Kriterium der Uneinigkeit")

    df_long = (
        df.melt(
            value_vars=[c for c in df.columns if c.startswith("typ_")],
            var_name="typ_col",
            value_name="note"
        )
        .dropna()
        .merge(
            typ_map[["typ_col", "typ_name"]],
            on="typ_col",
            how="left"
        )
    )

    MIN_BEW = 100

    std_ranking = (
        df_long
        .groupby("typ_name")
        .agg(
            mean_note=("note", "mean"),
            std_note=("note", "std"),
            anzahl=("note", "count")
        )
        .reset_index()
        .query("anzahl >= @MIN_BEW")
        .sort_values("std_note", ascending=False)
    )

    st.markdown("**Ranking nach Uneinigkeit (Standardabweichung)**")
    st.dataframe(
        std_ranking[["typ_name", "mean_note", "std_note", "anzahl"]],
        use_container_width=True
    )

    chart = (
        alt.Chart(std_ranking)
        .mark_bar()
        .encode(
            x=alt.X("std_note:Q", title="Standardabweichung"),
            y=alt.Y("typ_name:N", sort="-x", title="Bewertungskriterium"),
            tooltip=[
                alt.Tooltip("typ_name:N", title="Kriterium"),
                alt.Tooltip("mean_note:Q", title="Ø Note", format=".2f"),
                alt.Tooltip("std_note:Q", title="Std.-Abw.", format=".2f"),
                alt.Tooltip("anzahl:Q", title="Bewertungen")
            ]
        )
        .properties(height=420)
    )

    st.altair_chart(chart, use_container_width=True)

elif page == "Median der Kriterien":
    st.subheader("Median - Die typische Patientenbewertung")

    df_long = (
        df.melt(
            value_vars=[c for c in df.columns if c.startswith("typ_")],
            var_name="typ_col",
            value_name="note"
        )
        .dropna()
        .merge(
            typ_map[["typ_col", "typ_name"]],
            on="typ_col",
            how="left"
        )
    )

    MIN_BEW = 100

    median_ranking = (
        df_long
        .groupby("typ_name")
        .agg(
            median_note=("note", "median"),
            mean_note=("note", "mean"),
            anzahl=("note", "count")
        )
        .reset_index()
        .query("anzahl >= @MIN_BEW")
        .sort_values("median_note")
    )

    st.markdown("**Typische Bewertung je Kriterium (Median)**")
    st.dataframe(
        median_ranking,
        use_container_width=True
    )

    lollipop = (
        alt.Chart(median_ranking)
        .mark_line(strokeWidth=2)
        .encode(
            x=alt.X(
                "median_note:Q",
                title="Median der Bewertung",
                scale=alt.Scale(reverse=True)
            ),
            y=alt.Y(
                "typ_name:N",
                sort="-x",
                title="Bewertungskriterium"
            )
        )
        +
        alt.Chart(median_ranking)
        .mark_circle(size=120, color="#1f77b4")
        .encode(
            x=alt.X("median_note:Q"),
            y=alt.Y("typ_name:N", sort="-x"),
            tooltip=[
                alt.Tooltip("typ_name:N", title="Kriterium"),
                alt.Tooltip("median_note:Q", title="Median", format=".2f"),
                alt.Tooltip("mean_note:Q", title="Mittelwert", format=".2f"),
                alt.Tooltip("anzahl:Q", title="Bewertungen")
            ]
        )
    ).properties(height=450)

    st.altair_chart(lollipop, use_container_width=True)

    st.markdown("""
    **Interpretation:**  
    - Median zeigt die *ehrlichste* Bewertung  
    - Große Abweichung zwischen Mittelwert & Median → Ausreißer-Effekt  
    - Niedriger Median → stabil gute Wahrnehmung  
    """)
